# Sand Dune Comparison: TAM3C2 vs Mesh-Reference

Compare the two sand dune analyses that were produced by the sibling notebooks:

| | File | Method |
|---|---|---|
| **A** | `sand_dune_tam3c2.zip` | `sand_dune_analysis.ipynb` &mdash; TAM3C2 on noisy TLS XYZ point clouds |
| **B** | `sand_dune_obj_analysis.zip` | `sand_dune_reference.ipynb` &mdash; signed Z-displacement on clean OBJ meshes (ground truth) |

We treat **B** as the ground-truth reference and quantify how well **A** reproduces it.

**Sections**

1. Load both `SpatiotemporalAnalysis` archives.
2. Spatially align corepoints (A is sparse, B is denser &mdash; we match each A corepoint to its nearest B corepoint via a KDTree).
3. Per-epoch distance comparison: scatter, map of difference, statistics (RMSE / MAE / bias / correlation).
4. 4D-OBC comparison: plot both sets, spatial IoU, temporal IoU, spatio-temporal IoU.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.spatial import cKDTree, ConvexHull
from matplotlib.patches import Polygon

import py4dgeo


## 1. Load both analyses

In [ ]:
path_A = os.path.join(os.getcwd(), 'sand_dune_tam3c2.zip')          # TAM3C2 (A)
path_B = os.path.join(os.getcwd(), 'sand_dune_obj_analysis.zip')    # Mesh / signed Z (B)

analysis_A = py4dgeo.SpatiotemporalAnalysis(path_A)
analysis_B = py4dgeo.SpatiotemporalAnalysis(path_B)

cp_A = np.asarray(analysis_A.corepoints.cloud)
cp_B = np.asarray(analysis_B.corepoints.cloud)
d_A  = np.asarray(analysis_A.smoothed_distances if analysis_A.smoothed_distances is not None else analysis_A.distances)
d_B  = np.asarray(analysis_B.smoothed_distances if analysis_B.smoothed_distances is not None else analysis_B.distances)
obj_A = list(analysis_A.objects)
obj_B = list(analysis_B.objects)

print(f"A (TAM3C2)    : {cp_A.shape[0]:,} corepoints, distances shape {d_A.shape}, {len(obj_A)} OBCs")
print(f"B (Mesh ref)  : {cp_B.shape[0]:,} corepoints, distances shape {d_B.shape}, {len(obj_B)} OBCs")


## 2. Spatially align corepoints

A and B use different corepoint sets (different sampling on different inputs).
We build a KDTree on B's XY positions and, for every A corepoint, look up the
nearest B corepoint within `match_radius` meters. The result is a pair of index
arrays `(idx_A, idx_B)` of the **same length** that we use for every per-point
comparison below.


In [ ]:
match_radius = 0.5  # meters; tighten for stricter pairing

tree_B_xy = cKDTree(cp_B[:, :2])
dist_xy, nn_idx = tree_B_xy.query(cp_A[:, :2], k=1, distance_upper_bound=match_radius)

matched_mask = np.isfinite(dist_xy)
idx_A = np.where(matched_mask)[0]
idx_B = nn_idx[matched_mask]

print(f"Matched {len(idx_A):,} of {len(cp_A):,} A-corepoints "
      f"({len(idx_A)/len(cp_A)*100:.1f}%) within {match_radius} m")
print(f"Median XY pairing distance: {np.median(dist_xy[matched_mask]):.3f} m")


### Align the time axis

The two analyses may have a different number of epochs (e.g. if one of them
skipped some). We compare only the overlapping leading epochs.


In [ ]:
n_epochs = min(d_A.shape[1], d_B.shape[1])
print(f"Comparing first {n_epochs} epochs (A has {d_A.shape[1]}, B has {d_B.shape[1]})")

# Aligned distance matrices, shape (n_matched_corepoints, n_epochs)
dA = d_A[idx_A, :n_epochs]
dB = d_B[idx_B, :n_epochs]
diff = dA - dB  # A minus B (ground truth)


## 3. Distance comparison

### 3.1 Global statistics over all (matched corepoint, epoch) pairs


In [ ]:
valid = ~np.isnan(diff)
flat_diff = diff[valid]
flat_A = dA[valid]
flat_B = dB[valid]

n = flat_diff.size
bias   = float(np.mean(flat_diff))
mae    = float(np.mean(np.abs(flat_diff)))
rmse   = float(np.sqrt(np.mean(flat_diff**2)))
std    = float(np.std(flat_diff))
median = float(np.median(flat_diff))
corr   = float(np.corrcoef(flat_A, flat_B)[0, 1]) if n > 1 else float('nan')

print(f"N valid pairs    : {n:,}")
print(f"Bias (A - B)     : {bias:+.4f} m")
print(f"Median           : {median:+.4f} m")
print(f"MAE              : {mae:.4f} m")
print(f"RMSE             : {rmse:.4f} m")
print(f"Std of (A - B)   : {std:.4f} m")
print(f"Pearson r(A, B)  : {corr:.4f}")


### 3.2 Scatter plot A vs B and histogram of differences

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))

# Subsample for plotting if too many points
n_plot = min(50_000, flat_A.size)
sel = np.random.default_rng(0).choice(flat_A.size, size=n_plot, replace=False)

ax = axs[0]
ax.hexbin(flat_B[sel], flat_A[sel], gridsize=80, cmap='viridis', mincnt=1)
lim = max(abs(flat_A).max(), abs(flat_B).max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1, label='A = B')
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel('B: mesh signed Z [m] (ground truth)')
ax.set_ylabel('A: TAM3C2 distance [m]')
ax.set_title(f'A vs B  (Pearson r = {corr:.3f})')
ax.set_aspect('equal'); ax.legend(); ax.grid(alpha=0.3)

ax = axs[1]
ax.hist(flat_diff, bins=80, color='steelblue', edgecolor='k', alpha=0.8)
ax.axvline(0, color='k', lw=1)
ax.axvline(bias, color='r', lw=1.5, label=f'bias = {bias:+.3f}')
ax.axvline(median, color='orange', lw=1.5, label=f'median = {median:+.3f}')
ax.set_xlabel('A - B [m]')
ax.set_ylabel('count')
ax.set_title(f'Difference distribution  (RMSE = {rmse:.3f} m)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


### 3.3 Spatial map of mean / RMSE difference per corepoint

In [ ]:
# Per-corepoint summary (over the time axis)
per_cp_bias = np.nanmean(diff, axis=1)
per_cp_rmse = np.sqrt(np.nanmean(diff**2, axis=1))

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

ax = axs[0]
v = float(np.nanpercentile(np.abs(per_cp_bias), 95))
sc = ax.scatter(cp_A[idx_A, 0], cp_A[idx_A, 1], c=per_cp_bias,
                cmap='seismic_r', vmin=-v, vmax=v, s=6)
plt.colorbar(sc, ax=ax, label='mean(A - B) [m]')
ax.set_title('Per-corepoint bias map')
ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_aspect('equal')

ax = axs[1]
vmax = float(np.nanpercentile(per_cp_rmse, 95))
sc = ax.scatter(cp_A[idx_A, 0], cp_A[idx_A, 1], c=per_cp_rmse,
                cmap='magma', vmin=0, vmax=vmax, s=6)
plt.colorbar(sc, ax=ax, label='RMSE(A - B) [m]')
ax.set_title('Per-corepoint RMSE map')
ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_aspect('equal')

plt.tight_layout(); plt.show()


### 3.4 Per-epoch summary

In [ ]:
per_ep_bias = np.nanmean(diff, axis=0)
per_ep_rmse = np.sqrt(np.nanmean(diff**2, axis=0))
per_ep_mae  = np.nanmean(np.abs(diff), axis=0)

fig, ax = plt.subplots(figsize=(11, 4))
xs = np.arange(n_epochs)
ax.plot(xs, per_ep_bias, label='bias',  color='C0')
ax.plot(xs, per_ep_mae,  label='MAE',   color='C1')
ax.plot(xs, per_ep_rmse, label='RMSE',  color='C2')
ax.axhline(0, color='k', lw=0.7)
ax.set_xlabel('Epoch index (target)')
ax.set_ylabel('m')
ax.set_title('Per-epoch distance error of A relative to B')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 3.5 Example time series A vs B at a matched location

In [ ]:
# Pick a matched corepoint with non-trivial change in B
mag_B = np.nanmax(np.abs(dB), axis=1)
order = np.argsort(-mag_B)  # largest change first
sel_local = int(order[0])   # change index to inspect another location
print(f"Inspecting local pair {sel_local}: A cp {idx_A[sel_local]}, B cp {idx_B[sel_local]}")
print(f"  XY distance between paired corepoints: {dist_xy[idx_A[sel_local]]:.3f} m")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(dB[sel_local], label='B (mesh, ground truth)', color='black', lw=1.5)
ax.plot(dA[sel_local], label='A (TAM3C2)', color='C0', lw=1.2)
ax.fill_between(np.arange(n_epochs), dA[sel_local], dB[sel_local],
                alpha=0.25, color='red', label='difference')
ax.set_xlabel('Epoch index')
ax.set_ylabel('Signed displacement [m]')
ax.set_title(f'Time series at matched corepoint pair {sel_local}')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 3.6 Weighting comparison: Gaussian vs Linear vs None

The TAM3C2 distance written to `sand_dune_tam3c2.zip` was computed with the
default `weighting=GAUSSIAN`. Here we re-run the same TAM3C2 pipeline twice
more, with `weighting=LINEAR` and `weighting=NONE`, on **the same corepoints**
and **the same epoch series** as `sand_dune_analysis.ipynb`, and we compare
all three weighting schemes against the mesh ground truth B.

What we look at:

* Aggregate statistics (bias, MAE, RMSE, std, Pearson r) of `dA - dB` for each weighting.
* Per-epoch RMSE curve &mdash; does one weighting follow B more closely in time?
* Per-corepoint RMSE map &mdash; where in the scene does each weighting fail?

In [ ]:
# Re-load epochs to run TAM3C2 with different weighting schemes.
# (Only the distance matrix is stored in the .zip archive, so we have to
# recompute the distances from the raw point clouds.)
import re
from datetime import datetime
from py4dgeo import TAM3C2, Weighting, extract_reference_and_others
from py4dgeo.segmentation import temporal_averaging

data_path           = r'C:\rsa\research_proj\helios_simulation\output\Sand_Dune_tls'
reference_timestamp = datetime(2020, 1, 1, 6, 0, 0)

def _parse_ts(filename):
    m = re.search(r'(\d{8})_(\d{6})', os.path.basename(filename))
    if not m:
        return None
    try:
        return datetime.strptime(f"{m.group(1)}{m.group(2)}", '%Y%m%d%H%M%S')
    except (ValueError, IndexError):
        return None

def _load_epochs(folder):
    eps = []
    for fn in sorted(f for f in os.listdir(folder) if f.lower().endswith('.xyz')):
        ts = _parse_ts(fn)
        if ts is None:
            continue
        e = py4dgeo.read_from_xyz(os.path.join(folder, fn))
        e.timestamp = ts
        eps.append(e)
    return sorted(eps, key=lambda e: e.timestamp)

epochs_w = _load_epochs(data_path)
ref_w, others_w = extract_reference_and_others(epochs_w, reference_timestamp)
corepoints_w = np.asarray(analysis_A.corepoints.cloud)
print(f"Loaded {len(epochs_w)} epochs; reusing {len(corepoints_w):,} corepoints from A")

# TAM3C2 parameters mirror sand_dune_analysis.ipynb
tam_common = dict(
    epochs_timeseries=epochs_w,
    max_window_ratio=[0.2, 0.4],
    normal_radii=[0.5, 1.0],
    required_points=10,
    sigma_ratio=1.0,
    space_time_ratio=1.0,
    keep_neighborhoods=False,
    corepoints=corepoints_w,
    cyl_radius=0.5,
    max_distance=2.0,
    registration_error=0.01,
)

weighting_runs = {}  # name -> smoothed distances (n_cp_A, n_epochs)
for name, w in [('GAUSSIAN', Weighting.GAUSSIAN),
                ('LINEAR',   Weighting.LINEAR),
                ('NONE',     Weighting.NONE)]:
    print(f"Running TAM3C2 with weighting={name} ...")
    tam = TAM3C2(weighting=w, **tam_common)
    out_path = os.path.join(os.getcwd(), f'sand_dune_tam3c2_{name.lower()}.zip')
    ana = py4dgeo.SpatiotemporalAnalysis(out_path, force=True)
    ana.reference_epoch = ref_w
    ana.corepoints      = corepoints_w
    ana.m3c2            = tam
    ana.add_epochs(*others_w)
    ana.smoothed_distances = temporal_averaging(ana.distances, smoothing_window=3)
    weighting_runs[name] = np.asarray(ana.smoothed_distances)
    print(f"  distances shape = {weighting_runs[name].shape}")

In [ ]:
# Align every weighting run to the ground truth B (same idx_A / idx_B mapping
# as Section 2) and compute the aggregate statistics.
stats           = {}
per_cp_rmse_by_w = {}
per_ep_rmse_by_w = {}
per_ep_bias_by_w = {}


In [ ]:

for name, dM in weighting_runs.items():
    dA_w   = dM[idx_A, :n_epochs]
    diff_w = dA_w - dB
    v      = ~np.isnan(diff_w)
    fd, fA, fB = diff_w[v], dA_w[v], dB[v]
    stats[name] = dict(
        n     = int(fd.size),
        bias  = float(np.mean(fd)),
        mae   = float(np.mean(np.abs(fd))),
        rmse  = float(np.sqrt(np.mean(fd**2))),
        std   = float(np.std(fd)),
        corr  = float(np.corrcoef(fA, fB)[0, 1]) if fd.size > 1 else float('nan'),
    )
    per_cp_rmse_by_w[name] = np.sqrt(np.nanmean(diff_w**2, axis=1))
    per_ep_rmse_by_w[name] = np.sqrt(np.nanmean(diff_w**2, axis=0))
    per_ep_bias_by_w[name] = np.nanmean(diff_w, axis=0)

# Print a comparison table.
hdr = f"{'weighting':>10} | {'N':>9} | {'bias':>9} | {'MAE':>8} | {'RMSE':>8} | {'std':>8} | {'corr':>6}"
print(hdr)
print('-' * len(hdr))
for name, s in stats.items():
    print(f"{name:>10} | {s['n']:>9,} | {s['bias']:+9.4f} | {s['mae']:8.4f} | "
          f"{s['rmse']:8.4f} | {s['std']:8.4f} | {s['corr']:6.3f}")

# Bar chart of aggregate stats + per-epoch RMSE curves.
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))

metrics = ['bias', 'mae', 'rmse', 'std']
names   = list(stats)
x       = np.arange(len(metrics))
width   = 0.8 / len(names)
for i, name in enumerate(names):
    vals = [stats[name][m] for m in metrics]
    axs[0].bar(x + (i - (len(names) - 1) / 2) * width, vals, width, label=name)
axs[0].set_xticks(x); axs[0].set_xticklabels(metrics)
axs[0].axhline(0, color='k', lw=0.7)
axs[0].set_ylabel('m')
axs[0].set_title('Aggregate error vs ground truth B')
axs[0].legend(); axs[0].grid(alpha=0.3, axis='y')

for name in names:
    axs[1].plot(np.arange(n_epochs), per_ep_rmse_by_w[name], label=name, lw=1.5)
axs[1].set_xlabel('Epoch index (target)')
axs[1].set_ylabel('Per-epoch RMSE [m]')
axs[1].set_title('RMSE vs B as a function of target epoch')
axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Per-corepoint RMSE map for each weighting (shared colour scale).
all_rmse = np.concatenate([m[np.isfinite(m)] for m in per_cp_rmse_by_w.values()])
vmax     = float(np.nanpercentile(all_rmse, 95)) if all_rmse.size else 1.0

fig, axs = plt.subplots(1, len(per_cp_rmse_by_w),
                        figsize=(5 * len(per_cp_rmse_by_w), 4.5), squeeze=False)
xs, ys = cp_A[idx_A, 0], cp_A[idx_A, 1]
for ax, (name, m) in zip(axs[0], per_cp_rmse_by_w.items()):
    sc = ax.scatter(xs, ys, c=m, cmap='magma', vmin=0, vmax=vmax, s=6)
    plt.colorbar(sc, ax=ax, label='RMSE [m]')
    ax.set_title(f'Per-corepoint RMSE  -  weighting = {name}\n'
                 f'mean = {np.nanmean(m):.3f} m,   median = {np.nanmedian(m):.3f} m')
    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

# Bias map per weighting, signed colour scale.
vbias = float(np.nanpercentile(np.abs(np.concatenate(
    [np.nanmean(weighting_runs[n][idx_A, :n_epochs] - dB, axis=1) for n in names])), 95)) or 1e-3

fig, axs = plt.subplots(1, len(names), figsize=(5 * len(names), 4.5), squeeze=False)
for ax, name in zip(axs[0], names):
    bias_cp = np.nanmean(weighting_runs[name][idx_A, :n_epochs] - dB, axis=1)
    sc = ax.scatter(xs, ys, c=bias_cp, cmap='seismic_r', vmin=-vbias, vmax=vbias, s=6)
    plt.colorbar(sc, ax=ax, label='mean(A - B) [m]')
    ax.set_title(f'Per-corepoint bias  -  weighting = {name}')
    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 4. 4D-OBC comparison

We treat each 4D-OBC as a **spatio-temporal mask**: a set of corepoints, valid over an
epoch range `[start_epoch, end_epoch]`. To compare OBCs across the two analyses
(which have different corepoint sets), we first **rasterise** each OBC onto the
B corepoint indices via the matching from Section 2:

- For an OBC from A, take its corepoint indices, map them through `idx_A -> idx_B`
  to obtain a set of B-indices.
- For an OBC from B, its indices are already B-indices.

After that, both representations live in the same B-index space and can be
compared with standard set IoU. We then multiply by the temporal IoU
(intersection-over-union of the epoch ranges) to get a spatio-temporal IoU.


In [ ]:
# Pre-compute mapping A-cp-idx -> B-cp-idx for matched A corepoints only.
# Unmatched A corepoints have no counterpart and are dropped from any A-OBC.
a_to_b = {int(a): int(b) for a, b in zip(idx_A, idx_B)}

def obc_to_b_indices(obj, source):
    """Return the set of B-corepoint indices that an OBC occupies."""
    raw = np.asarray(obj.indices, dtype=int)
    if source == 'A':
        out = {a_to_b[i] for i in raw if i in a_to_b}
    else:  # 'B'
        out = set(raw.tolist())
    return out

def temporal_iou(a_start, a_end, b_start, b_end):
    lo = max(a_start, b_start)
    hi = min(a_end, b_end)
    inter = max(0, hi - lo + 1)
    union = (a_end - a_start + 1) + (b_end - b_start + 1) - inter
    return inter / union if union > 0 else 0.0

def spatial_iou(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union else 0.0

A_sets = [(obc_to_b_indices(o, 'A'), int(o.start_epoch), int(o.end_epoch)) for o in obj_A]
B_sets = [(obc_to_b_indices(o, 'B'), int(o.start_epoch), int(o.end_epoch)) for o in obj_B]

print(f"A: {len(A_sets)} OBCs   (size: min={min((len(s) for s,_,_ in A_sets), default=0)}, "
      f"max={max((len(s) for s,_,_ in A_sets), default=0)}, "
      f"median={int(np.median([len(s) for s,_,_ in A_sets])) if A_sets else 0})")
print(f"B: {len(B_sets)} OBCs   (size: min={min((len(s) for s,_,_ in B_sets), default=0)}, "
      f"max={max((len(s) for s,_,_ in B_sets), default=0)}, "
      f"median={int(np.median([len(s) for s,_,_ in B_sets])) if B_sets else 0})")


### 4.1 Pairwise IoU matrix and best matches

In [ ]:
nA, nB = len(A_sets), len(B_sets)
iou_spatial = np.zeros((nA, nB))
iou_temporal = np.zeros((nA, nB))
iou_st = np.zeros((nA, nB))

for i, (sa, sa_t0, sa_t1) in enumerate(A_sets):
    for j, (sb, sb_t0, sb_t1) in enumerate(B_sets):
        s = spatial_iou(sa, sb)
        t = temporal_iou(sa_t0, sa_t1, sb_t0, sb_t1)
        iou_spatial[i, j] = s
        iou_temporal[i, j] = t
        iou_st[i, j] = s * t

# Best B match for each A OBC
if nA and nB:
    best_j = iou_st.argmax(axis=1)
    best_iou = iou_st[np.arange(nA), best_j]
    print(f"For each A OBC, best spatio-temporal IoU with any B OBC:")
    print(f"  mean = {best_iou.mean():.3f}   median = {np.median(best_iou):.3f}   "
          f"max = {best_iou.max():.3f}")
    print(f"  A OBCs with ST-IoU > 0.1 : {(best_iou > 0.1).sum()} / {nA}")
    print(f"  A OBCs with ST-IoU > 0.3 : {(best_iou > 0.3).sum()} / {nA}")
    print(f"  A OBCs with ST-IoU > 0.5 : {(best_iou > 0.5).sum()} / {nA}")
else:
    best_j = np.array([], dtype=int)
    best_iou = np.array([])
    print('Cannot compute IoU: one of the OBC lists is empty.')


In [ ]:
# Visualise the IoU matrices
if nA and nB:
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, mat, title in zip(
        axs,
        [iou_spatial, iou_temporal, iou_st],
        ['Spatial IoU', 'Temporal IoU', 'Spatio-temporal IoU'],
    ):
        im = ax.imshow(mat, aspect='auto', vmin=0, vmax=max(mat.max(), 1e-6), cmap='viridis')
        plt.colorbar(im, ax=ax)
        ax.set_xlabel('B OBC index')
        ax.set_ylabel('A OBC index')
        ax.set_title(title)
    plt.tight_layout(); plt.show()


### 4.2 Spatial footprint of A and B OBCs on the same map

Each OBC is drawn as a filled convex hull. A in blue, B in red, overlapping
areas appear purple.


In [ ]:
def draw_obc_hulls(ax, obc_list, source, color):
    for o in obc_list:
        b_idx = list(obc_to_b_indices(o, source))
        if len(b_idx) < 3:
            continue
        pts = cp_B[b_idx, :2]
        try:
            hull = ConvexHull(pts)
        except Exception:
            continue
        ax.add_patch(Polygon(pts[hull.vertices], closed=True,
                             facecolor=color, edgecolor=color, alpha=0.25, lw=0.8))

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(cp_B[:, 0], cp_B[:, 1], s=0.4, c='lightgray', label='B corepoints')

draw_obc_hulls(ax, obj_A, 'A', 'blue')
draw_obc_hulls(ax, obj_B, 'B', 'red')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='blue', alpha=0.4, label=f'A OBCs ({len(obj_A)})'),
    Patch(facecolor='red',  alpha=0.4, label=f'B OBCs ({len(obj_B)})'),
], loc='upper right')
ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_aspect('equal')
ax.set_title('4D-OBC footprints (convex hulls): A (TAM3C2) vs B (Mesh)')
plt.tight_layout(); plt.show()


### 4.3 Inspect the best matched pair

In [ ]:
if nA and nB and best_iou.size and best_iou.max() > 0:
    a_i = int(np.argmax(best_iou))
    b_j = int(best_j[a_i])
    sa, sa_t0, sa_t1 = A_sets[a_i]
    sb, sb_t0, sb_t1 = B_sets[b_j]

    print(f"Best match: A[{a_i}] vs B[{b_j}]")
    print(f"  spatial IoU       = {iou_spatial[a_i, b_j]:.3f}")
    print(f"  temporal IoU      = {iou_temporal[a_i, b_j]:.3f}  "
          f"(A: {sa_t0}-{sa_t1}, B: {sb_t0}-{sb_t1})")
    print(f"  spatio-temp IoU   = {iou_st[a_i, b_j]:.3f}")
    print(f"  |A|={len(sa)}  |B|={len(sb)}  |A?B|={len(sa & sb)}  |A?B|={len(sa | sb)}")

    fig, axs = plt.subplots(1, 2, figsize=(14, 6))

    ax = axs[0]
    ax.scatter(cp_B[:, 0], cp_B[:, 1], s=0.4, c='lightgray')
    only_a = np.array(list(sa - sb), dtype=int)
    only_b = np.array(list(sb - sa), dtype=int)
    both   = np.array(list(sa & sb), dtype=int)
    if only_b.size: ax.scatter(cp_B[only_b, 0], cp_B[only_b, 1], s=6, c='red',    label='B only')
    if only_a.size: ax.scatter(cp_B[only_a, 0], cp_B[only_a, 1], s=6, c='blue',   label='A only')
    if both.size:   ax.scatter(cp_B[both,   0], cp_B[both,   1], s=8, c='purple', label='A ? B')
    ax.set_aspect('equal'); ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]')
    ax.set_title(f'Best pair footprint (spatial IoU = {iou_spatial[a_i, b_j]:.2f})')
    ax.legend(loc='upper right')

    ax = axs[1]
    ax.axvspan(sa_t0, sa_t1, color='blue', alpha=0.25, label=f'A [{sa_t0},{sa_t1}]')
    ax.axvspan(sb_t0, sb_t1, color='red',  alpha=0.25, label=f'B [{sb_t0},{sb_t1}]')
    ax.set_xlim(0, n_epochs - 1)
    ax.set_xlabel('Epoch index'); ax.set_yticks([])
    ax.set_title(f'Temporal extents (temporal IoU = {iou_temporal[a_i, b_j]:.2f})')
    ax.legend(loc='upper right')

    plt.tight_layout(); plt.show()
else:
    print('No overlapping OBCs to inspect.')


In [ ]:
if nA and nB and best_iou.size and best_iou.max() > 0:
    obj_a = obj_A[a_i]
    obj_b = obj_B[b_j]

    def plot_obc_ts(ax, d_mat, obj, member_color, title):
        idxs = np.asarray(obj.indices, dtype=int)
        seed_idx = int(obj.seed.index)
        s0, s1 = int(obj.start_epoch), int(obj.end_epoch)
        xs = np.arange(d_mat.shape[1])
        # subsample members so we draw at most ~200 lines
        step = max(1, len(idxs) // 200)
        for k in idxs[::step]:
            ax.plot(xs, d_mat[k], c=member_color, lw=0.5, alpha=0.6)
        ax.plot(xs, d_mat[seed_idx], c='black', lw=1.3, label='Seed timeseries')
        ax.axvspan(s0, s1, alpha=0.3, color='grey', label='4D-OBC timespan')
        ax.set_xlabel('Epoch index')
        ax.set_ylabel('Distance [m]')
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best')

    fig, axs = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    plot_obc_ts(
        axs[0], d_A, obj_a, 'blue',
        f'A[{a_i}] (TAM3C2)  |members|={len(obj_a.indices)}  seed CP={int(obj_a.seed.index)}',
    )
    plot_obc_ts(
        axs[1], d_B, obj_b, 'red',
        f'B[{b_j}] (Mesh ref)  |members|={len(obj_b.indices)}  seed CP={int(obj_b.seed.index)}',
    )
    plt.tight_layout()
    plt.show()
else:
    print('No overlapping OBCs to inspect.')

### 4.4 Summary table

In [ ]:
def summary_row(name, dvals):
    return (name,
            f"{np.nanmean(dvals):+.4f}",
            f"{np.nanmedian(dvals):+.4f}",
            f"{np.nanstd(dvals):.4f}")

print('Distance comparison')
print(f"  N pairs (matched corepoints): {len(idx_A):,}")
print(f"  N epochs compared           : {n_epochs}")
print(f"  Bias       : {bias:+.4f} m")
print(f"  MAE        : {mae:.4f} m")
print(f"  RMSE       : {rmse:.4f} m")
print(f"  Pearson r  : {corr:.4f}")
print()
print('4D-OBC comparison')
print(f"  # OBCs A / B             : {nA} / {nB}")
if best_iou.size:
    print(f"  mean best ST-IoU (A->B) : {best_iou.mean():.3f}")
    print(f"  A with ST-IoU > 0.5     : {(best_iou > 0.5).sum()} / {nA}")
    print(f"  A with ST-IoU > 0.3     : {(best_iou > 0.3).sum()} / {nA}")
    print(f"  A with ST-IoU > 0.1     : {(best_iou > 0.1).sum()} / {nA}")
